<a href="https://colab.research.google.com/github/tamfatkh/ebola/blob/main/Ebola.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import pymc as pm
import pytensor.tensor as pt
from pytensor.scan import scan
import arviz as az

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "DejaVu Serif", "Times New Roman"],
    "font.size": 14,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 15,
    "mathtext.fontset": "stix",
})

RANDOM_SEED = 12345
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
DATA_PATH = 'ebola_drc_sitrep_timeseries_en.csv'

def read_numbers_table(path, sheet_index=0, table_index=0):
    from numbers_parser import Document
    doc = Document(path)
    table = doc.sheets[sheet_index].tables[table_index]
    rows = []
    for r in range(table.num_rows):
        rows.append([table.cell(r, c).value for c in range(table.num_cols)])
    header = rows[0]
    return pd.DataFrame(rows[1:], columns=header)


def load_sitrep(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.numbers':
        df = read_numbers_table(path)
    elif ext in ['.csv', '.txt']:
        df = pd.read_csv(path)
    elif ext in ['.xlsx', '.xls']:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported data type: {ext}")
    return df

raw = load_sitrep(DATA_PATH)
raw.head()

In [ ]:
raw.columns = [str(c).strip() for c in raw.columns]

raw['report_date'] = pd.to_datetime(raw['report_date'], format='mixed')
for c in raw.columns:
    if c not in ['report_date', 'publication_date', 'source_filename']:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')

raw = raw.sort_values('report_date').reset_index(drop=True)


key_cols = [
    'cumulative_confirmed_cases', 'cumulative_confirmed_deaths',
    'cumulative_recovered', 'contact_followup_rate_pct',
    'patients_in_isolation_or_hospitalized', 'suspect_cases_today',
    'cumulative_suspect_cases', 'alerts_reported', 'alerts_investigated'
]

_esc_candidates = [c for c in raw.columns if 'escap' in c.lower() or 'evad' in c.lower()]
if _esc_candidates:
    raw.rename(columns={_esc_candidates[0]: 'escapees'}, inplace=True)
    raw['escapees'] = pd.to_numeric(raw['escapees'], errors='coerce')
else:
    raw['escapees'] = np.nan

raw.tail()

In [ ]:
start_date = raw['report_date'].min()
end_date = raw['report_date'].max()
all_dates = pd.date_range(start_date, end_date, freq='D')

df = pd.DataFrame({'report_date': all_dates})
df = df.merge(raw, on='report_date', how='left', suffixes=('', '_raw'))
df['t'] = (df['report_date'] - start_date).dt.days.astype(int)

df['cum_cases_obs'] = df['cumulative_confirmed_cases']
df['cum_deaths_obs'] = df['cumulative_confirmed_deaths']

if 'cumulative_recovered' in df.columns:
    df['cum_rec_obs'] = pd.to_numeric(df['cumulative_recovered'], errors='coerce')
else:
    df['cum_rec_obs'] = np.nan
    print('WARNING: no cumulative_recovered column -- recovery likelihood will be empty.')

df['new_confirmed_obs'] = df['cum_cases_obs'].diff()
df.loc[df['new_confirmed_obs'].isna(), 'new_confirmed_obs'] = df.loc[df['new_confirmed_obs'].isna(), 'cum_cases_obs']
df['new_confirmed_obs'] = df['new_confirmed_obs'].clip(lower=0)

df['new_deaths_obs'] = df['cum_deaths_obs'].diff()
df.loc[df['new_deaths_obs'].isna(), 'new_deaths_obs'] = df.loc[df['new_deaths_obs'].isna(), 'cum_deaths_obs']
df['new_deaths_obs'] = df['new_deaths_obs'].clip(lower=0)

if 'contact_followup_rate_pct' in df.columns:
    df['q_contact'] = df['contact_followup_rate_pct'] / 100.0
else:
    df['q_contact'] = np.nan


if 'suspect_cases_today' in df.columns and df['suspect_cases_today'].notna().any():
    df['symptom_proxy'] = df['suspect_cases_today']
else:
    df['symptom_proxy'] = np.nan

if 'cumulative_suspect_cases' in df.columns:
    sus_level = df['cumulative_suspect_cases'].astype(float).interpolate(
        method='linear', limit_direction='both'
    ).rolling(3, min_periods=1, center=True).mean()
    df['symptom_proxy'] = df['symptom_proxy'].fillna(sus_level)

if 'alerts_investigated' in df.columns:
    df['symptom_proxy'] = df['symptom_proxy'].fillna(df['alerts_investigated'])
if 'alerts_reported' in df.columns:
    df['symptom_proxy'] = df['symptom_proxy'].fillna(df['alerts_reported'])

if 'patients_in_isolation_or_hospitalized' in df.columns:
    df['H_obs_proxy'] = df['patients_in_isolation_or_hospitalized']
else:
    df['H_obs_proxy'] = np.nan

for c in ['q_contact', 'symptom_proxy', 'H_obs_proxy']:
    df[c] = df[c].astype(float).interpolate(limit_direction='both')

df['symptom_proxy'] = df['symptom_proxy'].fillna(0).clip(lower=0)
df['q_contact'] = df['q_contact'].clip(lower=0, upper=1).fillna(0.45)

log_sym = np.log1p(df['symptom_proxy'].values.astype(float))
df['z_symptom'] = (log_sym - np.nanmean(log_sym)) / (np.nanstd(log_sym) + 1e-8)

if 'escapees' in df.columns:
    df['escapees_obs'] = pd.to_numeric(df['escapees'], errors='coerce')
else:
    df['escapees_obs'] = np.nan

out_csv = 'ebola_drc_sitrep_cleaned_for_model.csv'
df.to_csv(out_csv, index=False)
df[['report_date','t','cum_cases_obs','cum_deaths_obs','new_confirmed_obs','symptom_proxy','q_contact','H_obs_proxy','escapees_obs']].head(12)


In [ ]:
esc_mask = df['escapees_obs'].notna() & df['H_obs_proxy'].notna()
esc_idx   = df.loc[esc_mask, 't'].values.astype('int64')
esc_y     = df.loc[esc_mask, 'escapees_obs'].values.astype('int64')
H_nat     = df.loc[esc_mask, 'H_obs_proxy'].values.astype('float64')
H_nat     = np.maximum(H_nat, 1.0)
rho_naive = esc_y.sum() / H_nat.sum()

In [ ]:
N_POP = 2_000_000.0
sigma_fixed = 1.0 / 7.65
T_U = 5.0
T_H = 10.8
nu = 1.0 / 2.0
nu_H = 1.0

obs_mask   = df['cum_cases_obs'].notna() & df['cum_deaths_obs'].notna()
obs_idx    = df.loc[obs_mask, 't'].values.astype('int64')
obs_dates  = df.loc[obs_mask, 'report_date'].values
cum_cases  = df.loc[obs_mask, 'cum_cases_obs'].values.astype('float64')
cum_deaths = df.loc[obs_mask, 'cum_deaths_obs'].values.astype('float64')

inc_cases_all  = np.diff(cum_cases)
inc_deaths_all = np.diff(cum_deaths)
win_lo  = obs_idx[:-1]
win_hi  = obs_idx[1:]
win_len = (win_hi - win_lo).astype('float64')

valid_inc = (inc_cases_all >= 0) & (inc_deaths_all >= 0)

inc_lo       = win_lo[valid_inc]
inc_hi       = win_hi[valid_inc]
inc_len      = win_len[valid_inc]
inc_dates    = obs_dates[1:][valid_inc]
y_inc_cases  = inc_cases_all[valid_inc].astype('int64')
y_inc_deaths = inc_deaths_all[valid_inc].astype('int64')

y_cases  = cum_cases
y_deaths = cum_deaths

rec_mask   = df['cum_rec_obs'].notna()
rec_t      = df.loc[rec_mask, 't'].values.astype('int64')
cum_rec    = df.loc[rec_mask, 'cum_rec_obs'].values.astype('float64')

if len(cum_rec) >= 2:
    assert cum_rec.max() > 1.5, (
        'cum_rec_obs looks like a proportion, not a count -- check the extraction.')

    inc_rec_all = np.diff(cum_rec)
    rlo_all, rhi_all = rec_t[:-1], rec_t[1:]
    valid_rec = inc_rec_all >= 0
    rec_lo    = rlo_all[valid_rec]
    rec_hi    = rhi_all[valid_rec]
    rec_len   = (rhi_all - rlo_all).astype('float64')[valid_rec]
    rec_dates = df.loc[rec_mask, 'report_date'].values[1:][valid_rec]
    y_inc_rec = inc_rec_all[valid_rec].astype('int64')
else:
    rec_lo = rec_hi = np.array([], dtype='int64')
    y_inc_rec = np.array([], dtype='int64')
    rec_len = np.array([], dtype='float64')
    rec_dates = np.array([])

T = int(df['t'].max()) + 1
q_contact_np = df['q_contact'].values.astype('float64')[:T]
z_symptom_np = df['z_symptom'].values.astype('float64')[:T]

C0_obs = float(y_cases[0])
D0_obs = float(y_deaths[0])

p_d_prior_median = 0.5
E0_guess = float(np.clip(
    y_inc_cases[0] / (p_d_prior_median * sigma_fixed * inc_len[0]),
    10.0, 500.0))
I0_guess = max(1.0, 0.15 * E0_guess)
n_dropped = int((~valid_inc).sum())

In [ ]:
def ebola_scan_sim(R0, eta, rho, a0, a1, b0, b1, E0, I0,
                   c_U, c_H, phi_D, m_F, delta,
                   q_contact, z_symptom):

    S0 = N_POP - E0 - I0 - C0_obs - D0_obs
    U0 = pt.as_tensor_variable(0.0)
    H0 = pt.as_tensor_variable(0.0)
    L0 = pt.as_tensor_variable(0.0)
    FU0 = pt.as_tensor_variable(0.0)
    FUD0 = pt.as_tensor_variable(0.0)
    FH0 = pt.as_tensor_variable(0.0)
    R0_state = pt.as_tensor_variable(0.0)
    C0 = pt.as_tensor_variable(C0_obs)
    D0 = pt.as_tensor_variable(D0_obs)
    Robs0 = pt.as_tensor_variable(R0_obs_init)

    y0 = pt.stack([S0, E0, U0, I0, H0, L0, FU0, FUD0, FH0, R0_state, C0, D0, Robs0])

    def dydt(y, qt, zt, t_day, R0_param, eta_param, rho_param,
             a0_param, a1_param, b0_param, b1_param,
             c_U_param, c_H_param, phi_D_param, m_F_param,
             delta_param):
        S, E, U, I, H, L, FU, FUD, FH, R, C, D, R_obs = y


        gamma_U = (1.0 - c_U_param) / T_U
        mu_U = c_U_param / T_U
        gamma_H = (1.0 - c_H_param) / T_H
        mu_H = c_H_param / T_H
        r_L = 1.0 / pt.maximum(T_H - 1.0 / (gamma_H + mu_H + rho_param), 1e-4)
        gamma_L = (1.0 - c_U_param) * r_L
        mu_L = c_U_param * r_L
        R_eff = R0_param * pt.exp(-delta_param * t_day)
        M0 = (
            1.0 / (gamma_U + mu_U)
            + (1.0 - phi_D_param) * m_F_param * (mu_U / (gamma_U + mu_U)) / nu
        )
        beta_U = R_eff / M0
        beta_F = m_F_param * beta_U

        N_eff = pt.maximum(S + E + U + I + H + L + FU + FUD + FH + R, 1.0)

        p_ct = pm.math.sigmoid(a0_param + a1_param * qt)
        p_sym = pm.math.sigmoid(b0_param + b1_param * zt)
        p_d = 1.0 - (1.0 - p_sym) * (1.0 - p_ct)
        p_d = pt.clip(p_d, 1e-5, 0.99999)
        lam = (beta_U * (U + I + L) + beta_F * FU) / N_eff

        new_exp = lam * S
        progress = sigma_fixed * E
        detect = p_d * progress
        undetect = (1.0 - p_d) * progress

        U_death = mu_U * U
        U_rec = gamma_U * U
        I_to_H = eta_param * I
        H_leak = rho_param * H
        H_death = mu_H * H
        H_rec = gamma_H * H
        L_death = mu_L * L
        L_rec = gamma_L * L
        unsafe_death = (1.0 - phi_D_param) * (U_death + L_death)
        safe_community_death = phi_D_param * (U_death + L_death)

        dS = -new_exp
        dE = new_exp - progress
        dU = undetect - U_death - U_rec
        dI = detect - I_to_H
        dH = I_to_H - H_leak - H_death - H_rec
        dL = H_leak - L_death - L_rec
        dFU = unsafe_death - nu * FU
        dFUD = safe_community_death - nu_H * FUD
        dFH = H_death - nu_H * FH
        dR = U_rec + H_rec + L_rec
        dC = detect
        dD = L_death + H_death
        dR_obs = H_rec

        return pt.stack([dS, dE, dU, dI, dH, dL, dFU, dFUD, dFH, dR, dC, dD, dR_obs])

    def one_step(qt, zt, t_day, y_prev,
                 R0_param, eta_param, rho_param,
                 a0_param, a1_param, b0_param, b1_param,
                 c_U_param, c_H_param, phi_D_param, m_F_param,
                 delta_param):
        dt = 1.0
        args = (R0_param, eta_param, rho_param, a0_param, a1_param, b0_param, b1_param,
                c_U_param, c_H_param, phi_D_param, m_F_param, delta_param)

        k1 = dydt(y_prev, qt, zt, t_day, *args)
        k2 = dydt(y_prev + 0.5 * dt * k1, qt, zt, t_day, *args)
        k3 = dydt(y_prev + 0.5 * dt * k2, qt, zt, t_day, *args)
        k4 = dydt(y_prev + dt * k3, qt, zt, t_day, *args)

        y_next = y_prev + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        return pt.maximum(y_next, 1e-9)

    T_steps = q_contact.shape[0]
    t_days = pt.arange(T_steps, dtype='float64')

    states, _ = scan(
        fn=one_step,
        sequences=[q_contact, z_symptom, t_days],
        outputs_info=y0,
        non_sequences=[R0, eta, rho, a0, a1, b0, b1, c_U, c_H, phi_D, m_F, delta],
        strict=True,
    )

    states = pt.concatenate([y0[None, :], states], axis=0)
    return states

In [ ]:
q_shared = pt.as_tensor_variable(q_contact_np)
z_shared = pt.as_tensor_variable(z_symptom_np)

R0_obs_init = float(cum_rec[0]) if len(y_inc_rec) > 0 else 0.0

with pm.Model() as ebola_model:

    R0 = pm.LogNormal('R0', mu=np.log(2.5), sigma=0.35)


    delta = pm.HalfNormal('delta', sigma=0.05)

    eta = pm.LogNormal('eta', mu=np.log(1/2), sigma=0.40)

    rho = pm.LogNormal('rho', mu=np.log(max(1e-3, rho_naive)), sigma=0.75)

    a0 = pm.Normal('a0_contact_intercept', mu=1.0, sigma=1.5)
    a1 = pm.HalfNormal('a1_contact_effect', sigma=2.0)

    b0 = pm.Normal('b0_symptom_intercept', mu=1.0, sigma=1.5)
    b1 = pm.HalfNormal('b1_symptom_effect', sigma=2.0)


    E0 = pm.LogNormal('E0', mu=np.log(E0_guess), sigma=1.10)
    I0 = pm.LogNormal('I0', mu=np.log(I0_guess), sigma=1.00)


    c_U = pm.Beta('c_U', alpha=1.0, beta=1.0)
    c_H = pm.Beta('c_H', alpha=1.0, beta=1.0)
    phi_D = pm.Beta('phi_D', alpha=24.0, beta=6.0)
    m_F = pm.LogNormal('m_F', mu=np.log(1.5), sigma=0.40)



    states = ebola_scan_sim(R0, eta, rho, a0, a1, b0, b1, E0, I0,
                            c_U, c_H, phi_D, m_F, delta, q_shared, z_shared)


    gamma_U_det = pm.Deterministic('gamma_U', (1.0 - c_U) / T_U)
    mu_U_det = pm.Deterministic('mu_U', c_U / T_U)
    gamma_H_det = pm.Deterministic('gamma_H', (1.0 - c_H) / T_H)
    mu_H_det = pm.Deterministic('mu_H', c_H / T_H)


    C_cum = states[:, 10]
    D_cum = states[:, 11]

    C_inc_hat = pm.Deterministic('C_inc_hat', C_cum[inc_hi] - C_cum[inc_lo])
    D_inc_hat = pm.Deterministic('D_inc_hat', D_cum[inc_hi] - D_cum[inc_lo])

    if len(y_inc_rec) > 0:
        R_cum = states[:, 12]
        R_inc_hat = pm.Deterministic('R_inc_hat', R_cum[rec_hi] - R_cum[rec_lo])

    C_hat = pm.Deterministic('C_hat', C_cum[obs_idx])
    D_hat = pm.Deterministic('D_hat', D_cum[obs_idx])

    t_shared = pt.arange(q_shared.shape[0], dtype='float64')
    R_eff_t = pm.Deterministic('R_eff_t', R0 * pt.exp(-delta * t_shared))
    p_ct_t = pm.Deterministic('p_ct_t', pm.math.sigmoid(a0 + a1 * q_shared))
    p_sym_t = pm.Deterministic('p_sym_t', pm.math.sigmoid(b0 + b1 * z_shared))
    p_d_t = pm.Deterministic('p_d_t', 1.0 - (1.0 - p_sym_t) * (1.0 - p_ct_t))

    invalpha_C = pm.HalfNormal('invalpha_C', sigma=1.0)
    invalpha_D = pm.HalfNormal('invalpha_D', sigma=1.0)
    alpha_C = pm.Deterministic('alpha_C', 1.0 / pt.maximum(invalpha_C, 1e-3))
    alpha_D = pm.Deterministic('alpha_D', 1.0 / pt.maximum(invalpha_D, 1e-3))
    alpha_ESC = pm.Gamma('alpha_ESC', alpha=2.0, beta=2.0)

    pm.NegativeBinomial('Y_inc_cases',
                        mu=pt.maximum(C_inc_hat, 1e-6),
                        alpha=alpha_C,
                        observed=y_inc_cases)
    pm.NegativeBinomial('Y_inc_deaths',
                        mu=pt.maximum(D_inc_hat, 1e-6),
                        alpha=alpha_D,
                        observed=y_inc_deaths)


    if len(y_inc_rec) > 0:
        invalpha_R = pm.HalfNormal('invalpha_R', sigma=1.0)
        alpha_R = pm.Deterministic('alpha_R', 1.0 / pt.maximum(invalpha_R, 1e-3))
        pm.NegativeBinomial('Y_inc_rec',
                            mu=pt.maximum(R_inc_hat, 1e-6),
                            alpha=alpha_R,
                            observed=y_inc_rec)


    H_nat_shared = pt.as_tensor_variable(H_nat)
    pm.NegativeBinomial('Y_escapees',
                        mu=pt.maximum(rho * H_nat_shared, 1e-6),
                        alpha=alpha_ESC,
                        observed=esc_y)

    idata = pm.sample(
        draws=1000, tune=1000, chains=4, cores=4,
        target_accept=0.95, random_seed=RANDOM_SEED,
        init='jitter+adapt_diag'
    )

az.summary(idata, var_names=['R0','eta','rho','delta','a0_contact_intercept','a1_contact_effect',
                             'b0_symptom_intercept','b1_symptom_effect','E0','I0',
                             'c_U','c_H','phi_D','m_F',
                             'alpha_C','alpha_D','alpha_R','alpha_ESC'],
            hdi_prob=0.95)

In [ ]:
from arviz.labels import MapLabeller

latex_labels = {
    'R0':                    r'$\mathcal{R}_0$',
    'eta':                   r'$\eta$',
    'rho':                   r'$\rho$',
    'delta':                 r'$\delta$',
    'a0_contact_intercept':  r'$a_0$ (contact intercept)',
    'a1_contact_effect':     r'$a_1$ (contact effect)',
    'b0_symptom_intercept':  r'$b_0$ (symptom intercept)',
    'b1_symptom_effect':     r'$b_1$ (symptom effect)',
    'E0':                    r'$E_0$',
    'I0':                    r'$I_0$',
    'c_U':                   r'$c_U$',
    'c_H':                   r'$c_H$',
    'phi_D':                 r'$\phi_D$',
    'm_F':                   r'$m_F$',
    'alpha_C':               r'$\alpha_C$',
    'alpha_D':               r'$\alpha_D$',
    'alpha_ESC':             r'$\alpha_{\mathrm{ESC}}$',
}

var_names = [
    'R0', 'eta', 'rho', 'delta',
    'a0_contact_intercept', 'a1_contact_effect',
    'b0_symptom_intercept', 'b1_symptom_effect',
    'E0', 'I0',
    'c_U', 'c_H', 'phi_D', 'm_F',
    'alpha_C', 'alpha_D', 'alpha_R', 'alpha_ESC',
]

labeller = MapLabeller(var_name_map=latex_labels)

az.plot_trace(idata, var_names=var_names, labeller=labeller)
plt.tight_layout()
plt.show()

In [ ]:
def posterior_interval(idata, name, hdi_prob=0.95):
    arr = idata.posterior[name].stack(sample=('chain', 'draw')).values
    med = np.median(arr, axis=-1)
    hdi = az.hdi(arr.T, hdi_prob=hdi_prob)
    return med, hdi[:, 0], hdi[:, 1]


def _plot_perday(dates, yobs, med, lo, hi, win_len, label,
                 per_day=True, color='tomato'):
    denom  = win_len if per_day else 1.0
    yobs_r = np.asarray(yobs) / denom
    med_r, lo_r, hi_r = med / denom, lo / denom, hi / denom

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(dates, yobs_r, s=22, marker='o', color='0.35', zorder=3,
               label='Observed')
    ax.plot(dates, med_r, '-', color=color, lw=1.8,
            label='Posterior median')
    ax.fill_between(dates, lo_r, hi_r,
                    alpha=0.25, color=color, label=r'95$\%$ HDI')
    ax.set_ylabel(f'New {label}')
    ax.set_xlabel('Report date')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(bottom=0)
    ax.legend()
    fig.tight_layout()
    plt.show()

for hat, yobs, dates, win_len, label, color in [
    ('C_inc_hat', y_inc_cases,  inc_dates, inc_len, 'confirmed cases',  'tomato'),
    ('D_inc_hat', y_inc_deaths, inc_dates, inc_len, 'confirmed deaths', 'firebrick'),
    ('R_inc_hat', y_inc_rec,    rec_dates, rec_len, 'discharges',       'seagreen'),
]:
    med, lo, hi = posterior_interval(idata, hat)
    _plot_perday(dates, yobs, med, lo, hi, win_len, label,
                 per_day=True, color=color)

rho_draws = idata.posterior['rho'].stack(sample=('chain', 'draw')).values
mu_esc    = rho_draws[None, :] * H_nat[:, None]
esc_med   = np.median(mu_esc, axis=1)
esc_hdi   = az.hdi(mu_esc.T, hdi_prob=0.95)
esc_dates = df.loc[esc_mask, 'report_date'].values

_plot_perday(esc_dates, esc_y, esc_med, esc_hdi[:, 0], esc_hdi[:, 1],
             win_len=None, label='Escapees', per_day=False, color='darkorange')

In [ ]:
p_d = idata.posterior['p_d_t'].stack(sample=('chain','draw')).values
p_sym = idata.posterior['p_sym_t'].stack(sample=('chain','draw')).values
p_ct = idata.posterior['p_ct_t'].stack(sample=('chain','draw')).values

summary_pd = pd.DataFrame({
    'report_date': df['report_date'].values[:T],
    'p_sym_median': np.median(p_sym, axis=1),
    'p_sym_low95': np.quantile(p_sym, 0.025, axis=1),
    'p_sym_high95': np.quantile(p_sym, 0.975, axis=1),
    'p_ct_median': np.median(p_ct, axis=1),
    'p_ct_low95': np.quantile(p_ct, 0.025, axis=1),
    'p_ct_high95': np.quantile(p_ct, 0.975, axis=1),
    'p_d_median': np.median(p_d, axis=1),
    'p_d_low95': np.quantile(p_d, 0.025, axis=1),
    'p_d_high95': np.quantile(p_d, 0.975, axis=1),
    'q_contact': q_contact_np,
    'symptom_proxy': df['symptom_proxy'].values[:T]
})
summary_pd.to_csv('posterior_detection_decomposition.csv', index=False)
summary_pd.head()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(summary_pd['report_date'], summary_pd['p_d_median'], label=r'combined $p_d(t)$')
plt.fill_between(summary_pd['report_date'], summary_pd['p_d_low95'], summary_pd['p_d_high95'], alpha=0.20)
plt.plot(summary_pd['report_date'], summary_pd['p_sym_median'], linestyle='--', label=r'symptom stream $p_{sym}(t)$')
plt.plot(summary_pd['report_date'], summary_pd['p_ct_median'], linestyle=':', label=r'contact stream $p_{ct}(t)$')
plt.ylabel('Detection probability')
plt.xlabel('Report date')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
log_scale_vars = {'R0', 'eta', 'rho', 'E0', 'I0', 'm_F', 'alpha_C', 'alpha_D', 'alpha_R', 'alpha_ESC'}

ncols = 3
nrows = -(-len(var_names_compare) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.2))
axes = axes.flatten()

for ax, var in zip(axes, var_names_compare):

    post = idata.posterior[var].values.flatten()

    pri  = idata.prior[var].values.flatten()

    use_log = var in log_scale_vars

    if use_log:
        post = np.log(post[post > 0])
        pri  = np.log(pri[pri > 0])
        xlabel = r'$\log\,$' + latex_labels.get(var, var)
    else:
        xlabel = latex_labels.get(var, var)

    both = np.concatenate([pri, post])
    lo, hi = np.percentile(both, [0.5, 99.5])
    bins = np.linspace(lo, hi, 61)
    ax.hist(pri,  bins=bins, density=True, alpha=0.40, color='steelblue', label='Prior')
    ax.hist(post, bins=bins, density=True, alpha=0.60, color='tomato',    label='Posterior')
    ax.set_xlim(lo, hi)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

for ax in axes[len(var_names_compare):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
from arviz.labels import MapLabeller

latex_labels = {
    "R0": r"$R_0$",
    "E0": r"$E_0$",
    "delta": r"$\delta$",
}

labeller = MapLabeller(var_name_map=latex_labels)

az.plot_pair(
    idata,
    var_names=["R0", "E0", "delta"],
    kind="kde",
    marginals=True,
    figsize=(9, 9),
    labeller=labeller,
)

plt.show()

In [ ]:
import numpy as np, matplotlib.pyplot as plt
try:
    import arviz as az
except Exception:
    az = None

T_U   = 5.0
T_H   = 10.8
NU    = 1.0/2.0
SIGMA = 1.0/7.65

def _get_idata():
    g = globals()
    if "idata" in g and g["idata"] is not None:
        return g["idata"]
    import os
    for p in ["./ebola_results/ebola_posterior_idata.nc", "ebola_posterior_idata.nc"]:
        if os.path.exists(p) and az is not None:
            try: return az.from_netcdf(p)
            except Exception: pass
    return None

IDATA = _get_idata()

_MEDIAN = dict(R0=3.804, eta=0.291, rho=0.035, delta=0.019,
               a0=0.953, a1=1.584, b0=0.890, b1=1.517,
               c_U=0.416, c_H=0.768, phi_D=0.803, m_F=1.602,
               E0=45.011, I0=8.708)

def posterior_draws(n=400, seed=0):

    names = ["R0","eta","rho","delta","c_U","c_H","phi_D","m_F","E0","I0",
             "a0","a1","b0","b1"]
    alias = {"a0":"a0_contact_intercept","a1":"a1_contact_effect",
             "b0":"b0_symptom_intercept","b1":"b1_symptom_effect"}
    if IDATA is not None:
        post = IDATA.posterior
        rng = np.random.default_rng(seed)
        stacked = post.stack(s=("chain","draw"))
        S = stacked.sizes["s"]
        idx = rng.choice(S, size=min(n,S), replace=False)
        out = {}
        for k in names:
            vn = alias.get(k,k)
            if vn in stacked:
                out[k] = np.asarray(stacked[vn].values)[..., idx].ravel()[:len(idx)]
            else:
                out[k] = np.full(len(idx), _MEDIAN[k])
        out["_n"] = len(idx); out["_source"]="posterior"
        return out
    out = {k: np.full(n, _MEDIAN[k]) for k in names}
    out["_n"]=n; out["_source"]="medians (idata not found -- bands are degenerate)"
    return out

_D = posterior_draws()

In [ ]:
def _rates(c_U, c_H, rho):
    gU=(1-c_U)/T_U; mU=c_U/T_U
    gH=(1-c_H)/T_H; mH=c_H/T_H
    rL=1.0/np.maximum(T_H-1.0/(gH+mH+rho), 1e-4)
    gL=(1-c_U)*rL; mL=c_U*rL
    return gU,mU,gH,mH,gL,mL

def M0(c_U, c_H, phi_D, m_F, rho):
    gU,mU,gH,mH,gL,mL=_rates(c_U,c_H,rho)
    return 1.0/(gU+mU) + (1-phi_D)*m_F*(mU/(gU+mU))/NU

def R_components(p_d, rho, phi_D, R0, eta, c_U, c_H, m_F, eta_L=0.0):

    gU,mU,gH,mH,gL,mL=_rates(c_U,c_H,rho)
    betaU = R0 / M0(c_U,c_H,phi_D,m_F,rho=0.0)
    betaF = m_F*betaU
    p_HL = rho/(rho+gH+mH)
    p_LH = eta_L/(gL+mL+eta_L)
    leak_surv = p_HL/(1.0 - p_HL*p_LH)
    tL = 1.0/(gL+mL+eta_L)
    mL_frac = mL/(gL+mL+eta_L)
    R_U = (1-p_d)*betaU/(gU+mU)
    R_I = p_d*betaU/eta
    R_L = p_d*leak_surv*betaU*tL
    R_F = (1-phi_D)*betaF/NU*((1-p_d)*mU/(gU+mU) + p_d*leak_surv*mL_frac)
    return R_U,R_I,R_L,R_F

def R_total(p_d, rho, phi_D, R0, eta, c_U, c_H, m_F, eta_L=0.0):
    return sum(R_components(p_d,rho,phi_D,R0,eta,c_U,c_H,m_F,eta_L))

_chk = R_total(0.0, 0.0, _MEDIAN["phi_D"], _MEDIAN["R0"], _MEDIAN["eta"],
               _MEDIAN["c_U"], _MEDIAN["c_H"], _MEDIAN["m_F"])
print(f"check R(p_d=0, rho=0) = {_chk:.3f}  (should equal R0 = {_MEDIAN['R0']:.3f})")
_r0 = R_total(0.5, _MEDIAN["rho"], _MEDIAN["phi_D"], _MEDIAN["R0"], _MEDIAN["eta"],
              _MEDIAN["c_U"], _MEDIAN["c_H"], _MEDIAN["m_F"], eta_L=0.0)
_r1 = R_total(0.5, _MEDIAN["rho"], _MEDIAN["phi_D"], _MEDIAN["R0"], _MEDIAN["eta"],
              _MEDIAN["c_U"], _MEDIAN["c_H"], _MEDIAN["m_F"], eta_L=0.2)
print(f"check R(eta_L=0)={_r0:.3f} >= R(eta_L=0.2)={_r1:.3f}  (re-isolation lowers R)")

In [ ]:
D = posterior_draws(n=400, seed=1)
def _band(f, xs):
    lo,mid,hi=[],[],[]
    for x in xs:
        v=np.asarray(f(x))
        h=az.hdi(v, hdi_prob=0.95)
        lo.append(float(h[0])); mid.append(float(np.median(v))); hi.append(float(h[1]))
    return map(np.array,(lo,mid,hi))

pd_grid  = np.linspace(0,1,60)
rho_grid = np.linspace(0,0.30,60)
phi_grid = np.linspace(0,1,60)
pd0, rho0, phi0 = 0.5, np.median(D["rho"]), np.median(D["phi_D"])

fig,ax=plt.subplots(1,3,figsize=(15,6),sharey=True)

f_pd = lambda x: R_total(x, D["rho"], phi0, D["R0"],D["eta"],D["c_U"],D["c_H"],D["m_F"])
lo,mid,hi=_band(f_pd,pd_grid)
ax[0].fill_between(pd_grid,lo,hi,alpha=.25); ax[0].plot(pd_grid,mid,lw=2)
ax[0].axhline(1,ls="--",c="k",lw=.8); ax[0].set_xlabel(r"$p_d$");
ax[0].set_ylabel(r"$\mathcal{R}(p_d)$")

f_rho= lambda x: R_total(pd0, x, phi0, D["R0"],D["eta"],D["c_U"],D["c_H"],D["m_F"])
lo,mid,hi=_band(f_rho,rho_grid)
ax[1].fill_between(rho_grid,lo,hi,alpha=.25,color="C1"); ax[1].plot(rho_grid,mid,lw=2,color="C1")
ax[1].axhline(1,ls="--",c="k",lw=.8); ax[1].axvline(rho0,ls=":",c="grey")
ax[1].set_xlabel(r"leakage $\rho$ (day$^{-1}$)");

f_phi= lambda x: R_total(pd0, D["rho"], x, D["R0"],D["eta"],D["c_U"],D["c_H"],D["m_F"])
lo,mid,hi=_band(f_phi,phi_grid)
ax[2].fill_between(phi_grid,lo,hi,alpha=.25,color="C2"); ax[2].plot(phi_grid,mid,lw=2,color="C2")
ax[2].axhline(1,ls="--",c="k",lw=.8)
ax[2].set_xlabel(r"death ascertainment $\phi_D$")
plt.tight_layout(); plt.show()

In [ ]:
med = {k:np.median(D[k]) for k in ["R0","eta","rho","c_U","c_H","m_F","phi_D"]}
RU,RI,RL,RF = R_components(pd_grid, med["rho"], med["phi_D"],
                           med["R0"],med["eta"],med["c_U"],med["c_H"],med["m_F"])
plt.figure(figsize=(8,6))
plt.stackplot(pd_grid, RU,RI,RL,RF,
              labels=[r"$R_U$ undetected",r"$R_I$ pre-isolation",r"$R_L$ leakage",r"$R_F$ funeral"],
              alpha=.85)
plt.plot(pd_grid, RU+RI+RL+RF, "k", lw=1.5, label=r"$\mathcal{R}(p_d)$")
plt.axhline(1,ls="--",c="k",lw=.8)
plt.xlabel(r"$p_d$"); plt.ylabel("contribution to $\\mathcal{R}$")
plt.legend(loc="upper right",fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
from matplotlib import cm

mF_draws = np.asarray(D["m_F"])
mF_med   = float(np.median(mF_draws))
mF_lo, mF_hi = az.hdi(mF_draws, hdi_prob=0.95)
print(f"m_F posterior: median={mF_med:.3f}, 95% HDI=[{mF_lo:.3f}, {mF_hi:.3f}]")
rho_hat = med["rho"]
rho_defs = [(r"$0\times\hat\rho$ (no leakage)", 0.0),
            (r"$\frac{1}{2}\hat\rho$",          0.5*rho_hat),
            (r"$\hat\rho$ (fitted)",            rho_hat),
            (r"$2\times\hat\rho$",              2*rho_hat),
            (r"$3\times\hat\rho$",              3*rho_hat)]
colors  = cm.viridis(np.linspace(0,0.9,len(rho_defs)))
ETA_MAX = 10.0
PD_MIN  = 0.7

PDc,PHIc = np.meshgrid(np.linspace(PD_MIN,1,120), np.linspace(PD_MIN,1,120))
print("R over the (p_d, phi_D) square at fitted eta -- why the old contour was blank:")
for lbl,rho_v in rho_defs:
    R = R_total(PDc, rho_v, PHIc, med["R0"],med["eta"],med["c_U"],med["c_H"], mF_med)
    print(f"  rho={rho_v:.4f} ({rho_v/rho_hat:.1f}x rho_hat): R in [{R.min():.2f}, {R.max():.2f}]"
          f"  ->  no R=1 curve (detection + burial alone cannot reach control)")

def pd_floor(rho, phi_D=None):
    phi_D = med["phi_D"] if phi_D is None else phi_D
    pd = np.linspace(PD_MIN,1,6001)
    r  = R_total(pd, rho, phi_D, med["R0"], 1e9, med["c_U"],med["c_H"], mF_med)
    below = pd[r<1]
    return float(below.min()) if len(below) else None

pd_ax   = np.linspace(PD_MIN,1,300)
etaS_ax = np.linspace(1,ETA_MAX,300)
PD, ES  = np.meshgrid(pd_ax, etaS_ax)

plt.figure(figsize=(8.4,6)); axp=plt.gca()

ctrl = np.zeros_like(PD)
for lbl,rho_v in rho_defs:
    ctrl += (R_total(PD, rho_v, med["phi_D"], med["R0"], med["eta"]*ES,
                     med["c_U"],med["c_H"], mF_med) < 1.0)
maxc = int(ctrl.max())
cf = axp.contourf(PD, ES, ctrl, levels=np.arange(-0.5, maxc+1.5),
                  cmap="Greens", alpha=0.45)

for (lbl,rho_v),col in zip(rho_defs,colors):
    Rmed = R_total(PD, rho_v, med["phi_D"], med["R0"], med["eta"]*ES, med["c_U"],med["c_H"], mF_med)
    axp.contour(PD, ES, Rmed, levels=[1.0], colors=[col], linewidths=2.3)
    fl = pd_floor(rho_v)
    if fl is not None:
        axp.axvline(fl, color=col, ls=":", lw=1.1)
        axp.plot([],[],color=col,lw=2.3, label=fr"{lbl}  ($p_d\!\geq${fl:.2f})")
    else:
        axp.plot([],[],color=col,lw=2.3,ls="--", label=fr"{lbl}  (uncontrollable)")

cbar = plt.colorbar(cf, ax=axp, ticks=range(0, maxc+1), pad=0.02)
cbar.ax.set_yticklabels(["none"] + [rho_defs[k][0] for k in range(maxc)])
cbar.set_label(r"control robustness: highest $\rho$ with $\mathcal{R}<1$")

axp.set_xlim(PD_MIN,1); axp.set_ylim(1,ETA_MAX); axp.axhline(1, ls=":", c="grey", lw=.7)
axp.set_xlabel(r"detection $p_d$"); axp.set_ylabel(r"isolation-speed factor $\kappa_\eta$")
axp.legend(title="leakage rate", fontsize=8, loc="lower left"); axp.grid(alpha=.2)
plt.tight_layout(); plt.show()

print("\nDetection floor (min p_d for control as eta->inf) and isolation speed at p_d=1:")
for lbl,rho_v in rho_defs:
    fl = pd_floor(rho_v)
    fp = lambda f: R_total(1.0, rho_v, med["phi_D"], med["R0"], med["eta"]*f, med["c_U"],med["c_H"], mF_med)-1
    fs = np.linspace(1,25,5001); v=fp(fs); s=np.where(np.diff(np.sign(v)))[0]
    at1 = (f"{fs[s[0]]:.2f}x (window {1/(med['eta']*fs[s[0]]):.2f} d)" if len(s)
           else "unreachable within 25x")
    floor_txt = f"p_d >= {fl:.3f}" if fl is not None else "NONE (uncontrollable at any p_d)"
    print(f"  rho={rho_v:.4f} ({rho_v/rho_hat:.1f}x rho_hat): floor {floor_txt:<34} | at p_d=1 need eta >= {at1}")

In [ ]:
def _get_covariates():
    g=globals()
    for qname,zname in [("q_contact","z_symptom"),("q_series","z_series"),("q_t","z_t")]:
        if qname in g and zname in g:
            return np.asarray(g[qname],float), np.asarray(g[zname],float)
    T = g.get("T_steps", g.get("T_days", 35))
    return np.full(int(T),0.452), np.zeros(int(T))

Q_T, Z_T = _get_covariates()
T_STEPS = len(Q_T)
N_POP = float(globals().get("N_POP", globals().get("N0", 1_000_000)))
print(f"T_steps={T_STEPS} | q_t in [{Q_T.min():.2f},{Q_T.max():.2f}] | N_POP={N_POP:,.0f}")

def simulate(params, *, rho_override=None, pd_mode=None, phi_override=None,
             eta_override=None, etaL=None, phi_hold_beta=False,T=None, dt=1.0):

    T = T or T_STEPS
    p=params
    def rho_at(t):
        if rho_override is None: return p["rho"]
        if callable(rho_override): return float(rho_override(t))
        return float(rho_override)
    def etaL_at(t):
        if etaL is None: return 0.0
        if callable(etaL): return float(etaL(t))
        return float(etaL)
    def eta_at(t):
        if eta_override is None: return p["eta"]
        if callable(eta_override): return float(eta_override(t))
        return float(eta_override)
    def pd_at(t, cU=None):
        ti = int(min(np.floor(t), T_STEPS-1)); qt = Q_T[ti]; zt = Z_T[ti]
        if pd_mode is not None and pd_mode[0]=="q_scale":
            qt = np.clip(qt*pd_mode[1], 0, 1)
        p_ct = 1/(1+np.exp(-(p["a0"]+p["a1"]*qt)))
        p_sym= 1/(1+np.exp(-(p["b0"]+p["b1"]*zt)))
        pd = 1-(1-p_sym)*(1-p_ct)
        if pd_mode is not None and pd_mode[0]=="const":
            pd = pd_mode[1]
        return np.clip(pd,1e-5,0.99999)
    phi = p["phi_D"] if phi_override is None else float(phi_override)

    phi_norm = p["phi_D"] if phi_hold_beta else phi

    gU=(1-p["c_U"])/T_U; mU=p["c_U"]/T_U
    gH=(1-p["c_H"])/T_H; mH=p["c_H"]/T_H
    M0_ = 1.0/(gU+mU) + (1-phi_norm)*p["m_F"]*(mU/(gU+mU))/NU
    def deriv(y,t):
        S,E,U,I,H,L,FU,FUD,FH,R,C,Dobs = y
        rho=rho_at(t); pd=pd_at(t); eL=etaL_at(t); eta_eff=eta_at(t)

        tauL = max(T_H - 1.0/(gH+mH+rho), 1.0/(gH+mH))
        rL = 1.0/tauL; gL=(1-p["c_U"])*rL; mL=p["c_U"]*rL
        R_eff=p["R0"]*np.exp(-p["delta"]*t)
        betaU=R_eff/M0_; betaF=p["m_F"]*betaU
        Neff=max(S+E+U+I+H+L+FU+FUD+FH+R,1.0)
        lam=(betaU*(U+I+L)+betaF*FU)/Neff
        new_exp=lam*S; prog=SIGMA*E; det=pd*prog; und=(1-pd)*prog
        dS=-new_exp; dE=new_exp-prog
        dU=und-gU*U-mU*U; dI=det-eta_eff*I
        dH=eta_eff*I-rho*H-gH*H-mH*H + eL*L
        dL=rho*H-gL*L-mL*L - eL*L
        dFU=(1-phi)*(mU*U+mL*L)-NU*FU
        dFUD=phi*(mU*U+mL*L)-1.0*FUD
        dFH=mH*H-1.0*FH
        dR=gU*U+gH*H+gL*L
        dC=det
        dDobs=phi*(mU*U+mL*L)+mH*H
        return np.array([dS,dE,dU,dI,dH,dL,dFU,dFUD,dFH,dR,dC,dDobs])

    y=np.zeros(12); y[1]=p["E0"]; y[3]=p["I0"]; y[0]=N_POP-p["E0"]-p["I0"]
    out=np.zeros((T,12)); true_inc=np.zeros(T)
    for t in range(T):
        out[t]=y
        k1=deriv(y,t); k2=deriv(y+dt/2*k1,t+dt/2)
        k3=deriv(y+dt/2*k2,t+dt/2); k4=deriv(y+dt*k3,t+dt)
        y=np.maximum(y+dt/6*(k1+2*k2+2*k3+k4),0.0)
        true_inc[t]=SIGMA*out[t,1]
    return dict(t=np.arange(T), C=out[:,10], Dobs=out[:,11],
                true_cuml=np.cumsum(true_inc), H=out[:,4], L=out[:,5], states=out)

def draws_list(n=200, seed=3):
    D=posterior_draws(n=n, seed=seed)
    keys=["R0","eta","rho","delta","c_U","c_H","phi_D","m_F","a0","a1","b0","b1","E0","I0"]
    return [{k:float(D[k][i]) for k in keys} for i in range(D["_n"])]

def band(scenario_fn, draws, key):
    M=np.array([scenario_fn(d)[key] for d in draws])
    h=az.hdi(M, hdi_prob=0.95)
    return h[:,0], np.median(M,0), h[:,1]

DRAWS = draws_list(n=150)
print("scenario draws:", len(DRAWS))

In [ ]:
rho_scenarios = {r"$0\times\hat\rho$ (no leakage)": 0.0,
                 r"$\frac{1}{2}\hat\rho$":          0.5,
                 r"$\hat\rho$ (fitted)":            None,
                 r"$2\times\hat\rho$":              2.0,
                 r"$3\times\hat\rho$":              3.0}
rho_plain = {r"$0\times\hat\rho$ (no leakage)": "0x rho (no leak)",
             r"$\frac{1}{2}\hat\rho$":           "0.5x rho",
             r"$\hat\rho$ (fitted)":             "fitted rho",
             r"$2\times\hat\rho$":               "2x rho",
             r"$3\times\hat\rho$":               "3x rho"}

def _run(d, m):
    return simulate(d) if m is None else simulate(d, rho_override=m*d["rho"])

keys=[("C","Cumulative reported cases"),
      ("Dobs","Cumulative reported deaths"),
      ("true_cuml","Cumulative actual infections")]

results={name:{k:band(lambda d,m=m: _run(d,m), DRAWS, k) for k,_ in keys}
         for name,m in rho_scenarios.items()}

fig,ax=plt.subplots(1,3,figsize=(15,6)); tt=np.arange(T_STEPS)
for name in rho_scenarios:
    for j,(k,ttl) in enumerate(keys):
        lo,mid,hi=results[name][k]
        ax[j].plot(tt,mid,lw=2,label=name); ax[j].fill_between(tt,lo,hi,alpha=.12)
        ax[j].set_ylabel(ttl); ax[j].set_xlabel("day")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"{'scenario':<18}{'rep.cases':>12}{'rep.deaths':>12}{'TRUE inf':>12}")
for name in rho_scenarios:
    c =results[name]["C"][1][-1]
    dd=results[name]["Dobs"][1][-1]
    ti=results[name]["true_cuml"][1][-1]
    print(f"{rho_plain[name]:<18}{c:>12,.0f}{dd:>12,.0f}{ti:>12,.0f}")

In [ ]:
weeks=[1,2,3,4]
fitted_rho=np.median(_D["rho"])
def step_rho(Xweeks, base):
    Xd=7*Xweeks
    return lambda t: base if t<Xd else 0.0
fig,ax=plt.subplots(1,3,figsize=(15,6)); tt=np.arange(T_STEPS)
base_fn=lambda d: simulate(d)
for j,(key,ttl) in enumerate([("C","REPORTED cases"),("Dobs","REPORTED deaths"),
                              ("true_cuml","TRUE infections")]):
    lo,mid,hi=band(base_fn,DRAWS,key)
    ax[j].plot(tt,mid,"k",lw=2.4,label="fitted ρ (no change)"); ax[j].fill_between(tt,lo,hi,alpha=.10,color="k")
for X in weeks:
    fn=lambda d,X=X: simulate(d, rho_override=step_rho(X, d["rho"]))
    for j,key in enumerate(["C","Dobs","true_cuml"]):
        lo,mid,hi=band(fn,DRAWS,key)
        ax[j].plot(tt,mid,lw=1.8,label=f"ρ→0 after wk {X}"); ax[j].fill_between(tt,lo,hi,alpha=.10)
        ax[j].axvline(7*X,ls=":",c="grey",lw=.6)
for a in ax: a.set_xlabel("day")
ax[0].legend(fontsize=8)
[a.set_ylabel(t) for a,t in zip(ax,["Cumulative reported cases","Cumulative reported deaths","Cumulative actual infections"])]
plt.tight_layout(); plt.show()

print("Averted TRUE infections vs fitted-ρ, by intervention week (posterior median):")
base_true=band(base_fn,DRAWS,"true_cuml")[1][-1]
for X in weeks:
    fn=lambda d,X=X: simulate(d, rho_override=step_rho(X,d["rho"]))
    ti=band(fn,DRAWS,"true_cuml")[1][-1]
    print(f"  ρ→0 after week {X}: {base_true-ti:,.0f} averted ({100*(base_true-ti)/base_true:.1f}%)")

In [ ]:
etaL_levels = {"no re-isolation":0.0, "$\eta_L=0.05/d$ (~20d)":0.05,
               "$\eta_L=0.10/d$ (~10d)":0.10, "$\eta_L=0.20/d$ (~5d)":0.20}
fig,ax=plt.subplots(1,3,figsize=(15,6)); tt=np.arange(T_STEPS)
for name,ev in etaL_levels.items():
    fn=lambda d,ev=ev: simulate(d, etaL=ev)
    for j,(key,ttl) in enumerate([("C","Cumulative reported cases"),
                                  ("Dobs","Cumulative reported deaths"),
                                  ("true_cuml","Cumulative actual infections")]):
        lo,mid,hi=band(fn,DRAWS,key)
        ax[j].plot(tt,mid,lw=2,label=name); ax[j].fill_between(tt,lo,hi,alpha=.10)
        ax[j].set_ylabel(ttl); ax[j].set_xlabel("day")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

base_true=band(lambda d: simulate(d, etaL=0.0), DRAWS, "true_cuml")[1][-1]
print("Averted TRUE infections vs no re-isolation (posterior median):")
for name,ev in etaL_levels.items():
    if ev==0.0: continue
    ti=band(lambda d,ev=ev: simulate(d, etaL=ev), DRAWS, "true_cuml")[1][-1]
    print(f"  {name:<22}: {base_true-ti:>7,.0f} averted ({100*(base_true-ti)/base_true:5.1f}%)")

def step_etaL(Xweeks, val=0.20):
    Xd=7*Xweeks
    return lambda t: (val if t>=Xd else 0.0)
print("\nStep re-isolation eta_L=0.20 switched on after week X (averted vs base):")
for X in [1,2,3,4]:
    ti=band(lambda d,X=X: simulate(d, etaL=step_etaL(X)), DRAWS, "true_cuml")[1][-1]
    print(f"  on after week {X}: {base_true-ti:>7,.0f} averted ({100*(base_true-ti)/base_true:5.1f}%)")

In [ ]:
eta_scales = {"fitted $\eta$ (1x)":1.0, "1.5x":1.5, "2x":2.0, "3x":3.0}
fig,ax=plt.subplots(1,3,figsize=(15,6)); tt=np.arange(T_STEPS)
for name,sc in eta_scales.items():
    fn=lambda d,sc=sc: simulate(d, eta_override=sc*d["eta"])
    for j,(key,ttl) in enumerate([("C","Cumulative reported cases"),
                                  ("Dobs","Cumulative reported deaths"),
                                  ("true_cuml","Cumulative actual infections")]):
        lo,mid,hi=band(fn,DRAWS,key)
        ax[j].plot(tt,mid,lw=2,label=name); ax[j].fill_between(tt,lo,hi,alpha=.10)
        ax[j].set_ylabel(ttl); ax[j].set_xlabel("day")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

base_true=band(lambda d: simulate(d), DRAWS, "true_cuml")[1][-1]
print("Averted TRUE infections vs fitted eta (posterior median):")
for name,sc in eta_scales.items():
    if sc==1.0: continue
    ti=band(lambda d,sc=sc: simulate(d, eta_override=sc*d["eta"]), DRAWS, "true_cuml")[1][-1]
    print(f"  eta {name:<6}: {base_true-ti:>7,.0f} averted ({100*(base_true-ti)/base_true:5.1f}%)")


In [ ]:
order = sorted(curves, key=lambda n: curves[n]["mid"][-1], reverse=True)

with plt.rc_context({"font.size":16, "axes.titlesize":16, "axes.labelsize":16,
                     "xtick.labelsize":16, "ytick.labelsize":16, "legend.fontsize":14}):
    fig,ax=plt.subplots(1,2,figsize=(17,7),gridspec_kw={"width_ratios":[2.4,1]})
    handles={}
    for name,c in curves.items():
        (ln,)=ax[0].plot(c["xn"],c["mid"],lw=2.4,color=cols[name],label=name)
        handles[name]=ln
        ax[0].fill_between(c["xn"],c["lo"],c["hi"],color=cols[name],alpha=.12)
        if c["has_knee"]:
            k=c["k"]; ax[0].plot(c["xn"][k],c["mid"][k],"o",color=cols[name],ms=9,mec="k",zorder=5)
            ax[0].annotate(f"knee ~{c['xs'][k]:.2g}", (c["xn"][k],c["mid"][k]),
                           textcoords="offset points", xytext=(6,-14), fontsize=13, color=cols[name])
    ax[0].set_xlabel(r"lever intensity (normalized: $1=q_t\!\times\!3,\ \eta\!\times\!3,\ "
                     r"\eta_L\!=\!0.20,\ \rho\!\to\!0,\ \phi_D\!\to\!1$)")
    ax[0].set_ylabel("Cumulative actual infections averted (%)")
    ax[0].legend([handles[n] for n in order], order, fontsize=13)
    ax[0].grid(alpha=.2)

    names=order
    ceil=[curves[n]["mid"][-1] for n in names]
    elo =[curves[n]["mid"][-1]-curves[n]["lo"][-1] for n in names]
    ehi =[curves[n]["hi"][-1]-curves[n]["mid"][-1] for n in names]
    ax[1].barh(range(len(names)), ceil, xerr=[elo,ehi], color=[cols[n] for n in names], alpha=.85,
               error_kw=dict(ecolor="k",capsize=3,lw=1))
    ax[1].set_yticks(range(len(names))); ax[1].set_yticklabels(names,fontsize=13)
    ax[1].invert_yaxis(); ax[1].set_xlabel("averted at full intensity (%)")
    ax[1].grid(alpha=.2,axis="x")
    plt.tight_layout(); plt.show()